**Задание 1-2**

---

In [5]:
# ==========================================
# ЗАДАНИЕ 1 и 2: Экспертная система "Деревья"
# ==========================================

print("--- ЗАДАНИЕ 1 и 2: Определение породы дерева ---")

# 1. БАЗА ЗНАНИЙ: ФАКТЫ (Вопросы)
fact = {
    1: 'порода лиственная', 
    2: 'порода хвойная', 
    3: 'древесина по прочности мягкая', 
    4: 'древесина по прочности твердая', 
    5: 'древесина по прочности очень твердая', 
    6: 'древесина по цвету серо-коричневая', 
    7: 'древесина по цвету светло-красная', 
    8: 'древесина по цвету светлая', 
    9: 'древесина по цвету темная', 
    10: 'древесина по составу смолистая', 
    11: 'древесина по составу очень смолистая', 
    12: 'текстура у древесины крупная', 
    13: 'текстура у древесины мелкая',
    # Новые факты (Задание 2)
    14: 'кора имеет белый цвет', 
    15: 'древесина крайне стойкая к гниению'
}

# 2. БАЗА ЗНАНИЙ: ПРАВИЛА (Цели)
rules = {
    'Бук': [1, 4, 7, 12], 
    'Дуб': [1, 4, 6, 13], 
    'Ель': [2, 3, 8, 10], 
    'Осина': [1, 3, 8, 13], 
    'Сосна': [2, 3, 8, 11],
    'Тис': [1, 5, 9],
    # Новые правила (Задание 2)
    'Береза': [1, 3, 8, 14],       
    'Лиственница': [2, 4, 8, 15]   
}

# 3. РАБОЧАЯ ОБЛАСТЬ
check = rules.copy()
for k, v in check.items():
    check[k] = [0] * len(v)

# --- ФУНКЦИИ (из методички) ---

def question(n):
    if n not in fact: return None
    print(f"Вопрос: {fact[n]}?")
    while True:
        ans = input("Напишите: да или нет: ").lower().strip()
        if ans in ['да', 'нет']: return ans
        print("Введите корректно: 'да' или 'нет'.")

def process(n, answer):
    global check
    if answer == 'да':
        for k, v in rules.items():
            if n in v and k in check:
                check[k][v.index(n)] = 1
    elif answer == 'нет':
        # Удаляем только те правила, где этот факт обязателен
        keys_to_delete = [k for k, v in rules.items() if n in v and k in check]
        for k in keys_to_delete: check.pop(k)
    # Сортировка по сумме совпадений
    check = dict(sorted(check.items(), key=lambda item: -sum(item[1])))

def all_true(k):
    return k in check and all(check[k])

# --- ГЛАВНЫЙ ЦИКЛ (Задание 1) ---
def run_tree_es():
    global check
    check = rules.copy()
    for k, v in check.items(): check[k] = [0] * len(v)
    
    asked = set()
    while check:
        found = False
        for k in check:
            if all_true(k):
                print(f"\n✅ РЕЗУЛЬТАТ: Это дерево - {k}!")
                found = True
                break
        if found: break

        best_key = next(iter(check))
        next_q = None
        for fid in rules[best_key]:
            if fid not in asked and check[best_key][rules[best_key].index(fid)] == 0:
                next_q = fid
                break
        
        if next_q is None:
            check.pop(best_key)
            continue
            
        asked.add(next_q)
        ans = question(next_q)
        process(next_q, ans)

    if not check:
        print("\n❌ Ответа ЭС не знает (нет совпадений в базе).")

# Запуск
run_tree_es()

--- ЗАДАНИЕ 1 и 2: Определение породы дерева ---
Вопрос: порода лиственная?
Вопрос: древесина по прочности твердая?
Вопрос: древесина по цвету светло-красная?
Вопрос: древесина по цвету серо-коричневая?
Вопрос: древесина по прочности мягкая?
Вопрос: древесина по прочности очень твердая?
Вопрос: древесина по цвету темная?

✅ РЕЗУЛЬТАТ: Это дерево - Тис!


In [ ]:
# ==========================================
# ЗАДАНИЕ 3: Экспертная система "Конфигурация ПК"
# ==========================================

print("\n--- ЗАДАНИЕ 3: Подбор конфигурации ПК ---")

# 1. БАЗА ЗНАНИЙ: ФАКТЫ
# Факты сгруппированы так, чтобы ответ "нет" не ломал систему
fact_pc = {
    # Цели (1 из 3)
    1: 'Основная цель: Игры и развлечения',
    2: 'Основная цель: Работа (офис, документы, браузер)',
    3: 'Основная цель: Профессиональная работа (монтаж, 3D, код)',
    
    # Бюджет (1 из 3)
    4: 'Бюджет: Эконом (до 50 000 руб.)',
    5: 'Бюджет: Средний (50 000 - 100 000 руб.)',
    6: 'Бюджет: Высокий (более 100 000 руб.)',
    
    # Производитель (не во всех правилах, чтобы не ломать систему)
    7: 'Предпочтение: Intel (процессоры)',
    8: 'Предпочтение: AMD (процессоры)',
    9: 'Предпочтение: NVIDIA (видеокарты)',
    
    # Тип устройства
    10: 'Тип: Ноутбук',
    11: 'Тип: Стационарный компьютер'
}

# 2. БАЗА ЗНАНИЙ: ПРАВИЛА
# Дублируем правила для разных производителей, чтобы ответ "нет" не удалял всё
rules_pc = {
    # Игровые (ПК)
    'Игровой ПК начальный (AMD)': [1, 4, 8, 11],
    'Игровой ПК средний (Intel)': [1, 5, 7, 11],
    'Игровой ПК топ (Intel+NVIDIA)': [1, 6, 7, 9, 11],
    
    # Игровые (Ноутбуки)
    'Игровой ноутбук (средний)': [1, 5, 7, 10],
    'Игровой ноутбук (топ)': [1, 6, 9, 10],
    
    # Офисные
    'Офисный ПК (бюджетный)': [2, 4, 11],
    'Офисный ПК (стандарт)': [2, 5, 11],
    'Офисный ноутбук': [2, 5, 10],
    
    # Профессиональные
    'Рабочая станция (монтаж/3D)': [3, 6, 7, 9, 11],
    'Ноутбук для разработчика': [3, 6, 10],
}

# 3. РАБОЧАЯ ОБЛАСТЬ
check_pc = rules_pc.copy()
for k, v in check_pc.items():
    check_pc[k] = [0] * len(v)

# --- ФУНКЦИИ ---

def question_pc(n):
    if n not in fact_pc: return None
    print(f"Вопрос: {fact_pc[n]}?")
    while True:
        ans = input("Напишите: да или нет: ").lower().strip()
        if ans in ['да', 'нет']: return ans
        print("Введите 'да' или 'нет'.")

def process_pc(n, answer):
    global check_pc
    if answer == 'да':
        for k, v in rules_pc.items():
            if n in v and k in check_pc:
                check_pc[k][v.index(n)] = 1
    elif answer == 'нет':
        keys_to_delete = [k for k, v in rules_pc.items() if n in v and k in check_pc]
        for k in keys_to_delete: check_pc.pop(k)
    check_pc = dict(sorted(check_pc.items(), key=lambda item: -sum(item[1])))

def all_true_pc(k):
    return k in check_pc and all(check_pc[k])

# --- ГЛАВНЫЙ ЦИКЛ ---
def run_pc_es():
    global check_pc
    check_pc = rules_pc.copy()
    for k, v in check_pc.items(): check_pc[k] = [0] * len(v)
    
    asked = set()
    while check_pc:
        found = False
        for k in check_pc:
            if all_true_pc(k):
                print(f"\n✅ РЕКОМЕНДАЦИЯ: {k}")
                found = True
                break
        if found: break

        best_key = next(iter(check_pc))
        next_q = None
        for fid in rules_pc[best_key]:
            if fid not in asked and check_pc[best_key][rules_pc[best_key].index(fid)] == 0:
                next_q = fid
                break
        
        if next_q is None:
            check_pc.pop(best_key)
            continue
            
        asked.add(next_q)
        ans = question_pc(next_q)
        process_pc(next_q, ans)

    if not check_pc:
        print("\n❌ Конфигурация не найдена.")
        print("💡 Попробуйте изменить ответы (например, выберите другой бюджет).")

# Запуск
run_pc_es()


--- ЗАДАНИЕ 3: Подбор конфигурации ПК ---
Вопрос: Основная цель: Игры и развлечения?
Вопрос: Бюджет: Эконом (до 50 000 руб.)?
Вопрос: Предпочтение: AMD (процессоры)?
Вопрос: Предпочтение: NVIDIA (видеокарты)?
Вопрос: Тип: Стационарный компьютер?

✅ РЕКОМЕНДАЦИЯ: Игровой ПК начального уровня
